# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_json = dataset.metadata.to_json()
print(f"{getattr(dataset.metadata, 'name', 'Dataset')}: {getattr(dataset.metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record sets and their fields by @id
record_set_ids = [rs['@id'] for rs in metadata_json.get('recordSet', [])]
print(f"Record Sets (@id): {record_set_ids}\n")

record_set_fields = {}
for recset in metadata_json.get('recordSet', []):
    rid = recset['@id']
    fields = recset.get('field', [])
    # Handle both list and single dict
    field_ids = [f['@id'] if isinstance(f, dict) else f for f in fields] if isinstance(fields, list) else ([fields['@id']] if isinstance(fields, dict) else [])
    record_set_fields[rid] = field_ids
    print(f"Record Set: {rid}")
    print(f"  Fields (@id): {field_ids}\n")

# Print a sample of one record from each record set by @id
for rid in record_set_ids:
    print(f"Sample record from record set {rid}:")
    records_iter = dataset.records(record_set=rid)
    try:
        print(next(records_iter))
    except StopIteration:
        print("  No records found.")
    except Exception as e:
        print(f"  Error: {e}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Record set selection (substitute with the @id you want to analyze; here we choose the first one if available)
record_sets = record_set_ids
dataframes = {}

selected_record_set = None
if record_sets:
    selected_record_set = record_sets[0]

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from {record_set_id}")
    except Exception as e:
        print(f"Error loading records from {record_set_id}: {e}")
        dataframes[record_set_id] = pd.DataFrame()

if selected_record_set and not dataframes[selected_record_set].empty:
    print(f"Columns in record set {selected_record_set}:\n", dataframes[selected_record_set].columns.tolist())
    display(dataframes[selected_record_set].head())
else:
    print(f"No records found for record set: {selected_record_set}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

if selected_record_set and not dataframes[selected_record_set].empty:
    df = dataframes[selected_record_set]
    
    # Try to auto-select a numeric field for demonstration (by selecting first float/int column)
    numeric_field = None
    for c in df.columns:
        # Try converting to numeric and see if field has at least one real value
        try:
            vals = pd.to_numeric(df[c], errors='coerce')
            if vals.notnull().sum() > 0:
                numeric_field = c
                df[c] = vals
                break
        except Exception:
            continue

    if numeric_field is not None:
        print(f"Selected numeric field: {numeric_field}")
        threshold = np.nanpercentile(df[numeric_field], 75) if df[numeric_field].notnull().sum() >= 4 else df[numeric_field].mean()
        
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to select a group field (categorical string column) for demonstration
        group_field = None
        for c in df.columns:
            if c != numeric_field and df[c].dtype == object and df[c].nunique() > 1 and df[c].notnull().sum() > 4:
                group_field = c
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data by {group_field} and mean {numeric_field}:")
            display(grouped_df.head())
        else:
            print("No categorical group field found for groupby analysis.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if selected_record_set and not dataframes[selected_record_set].empty:
    df = dataframes[selected_record_set]
    if 'numeric_field' in locals() and numeric_field is not None:
        fig, ax = plt.subplots(figsize=(8, 5))
        df[numeric_field].hist(bins=20, ax=ax)
        ax.set_title(f'Distribution of {numeric_field}')
        ax.set_xlabel(numeric_field)
        ax.set_ylabel('Frequency')
        plt.show()

        # Boxplot by group if exists
        if 'group_field' in locals() and group_field and group_field in df.columns:
            plt.figure(figsize=(10, 6))
            df[[group_field, numeric_field]].boxplot(by=group_field)
            plt.title(f'{numeric_field} by {group_field}')
            plt.suptitle('')
            plt.xlabel(group_field)
            plt.ylabel(numeric_field)
            plt.show()
    else:
        print('No suitable numeric field for visualization.')
else:
    print('No data available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the Croissant schema and displayed the dataset and record set structure.
- We demonstrated how to extract record sets and their fields using their `@id` references, as required by the Croissant standard.
- Exploratory analysis and visualization steps were performed using data from fields identified dynamically as numeric or categorical, demonstrating filtering, normalization, grouping, and plotting.

Next steps could be deeper domain-specific analysis of protein records, feature engineering, or modeling.